# Experimento D (Fase 3): close_mosaic en los últimos 15 epochs

Desactiva el mosaico en los últimos 15 epochs para afinar el modelo con imágenes en su distribución natural. Hiperparámetros en `configs/hyp_exp_D.yaml`.

> **Notebook delgado**: solo orquesta. La lógica vive en `src/` del repositorio.

In [ ]:
# Setup en Kaggle: clona el repo del proyecto e instala dependencias.
# El código pesado vive en `src/`; este notebook solo orquesta.
import os, subprocess, sys

REPO_URL = "https://github.com/PercyTechX/epp-yolo-compliance.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Carga el archivo de hiperparámetros del experimento (hyp_exp_D.yaml).
# Sobrescribe `path` de data.yaml para apuntar al Kaggle Dataset montado.
# La ruta se auto-detecta porque Kaggle a veces monta los datasets
# bajo /kaggle/input/<slug>/ y otras bajo /kaggle/input/datasets/<user>/<slug>/.
import os, yaml
from pathlib import Path


def find_dataset_root() -> str:
    """Localiza la raíz del Kaggle Dataset buscando images/ + labels/."""
    candidates = [
        "/kaggle/input/epp-yolo-processed",
        "/kaggle/input/datasets/jpnazariop/epp-yolo-processed",
    ]
    for c in candidates:
        if os.path.isdir(f"{c}/images/train"):
            return c
    # Fallback: escaneo recursivo de /kaggle/input/.
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "images" in dirs and "labels" in dirs:
            if os.path.isdir(f"{root}/images/train"):
                return root
    raise RuntimeError("Dataset no encontrado bajo /kaggle/input/")


HYP_FILE = Path("configs/hyp_exp_D.yaml")
DATA_YAML = Path("configs/data.yaml")
KAGGLE_DATA_ROOT = find_dataset_root()
print(f"Dataset montado en: {KAGGLE_DATA_ROOT}")

with HYP_FILE.open() as f:
    hyp = yaml.safe_load(f)

data_cfg = yaml.safe_load(DATA_YAML.read_text())
data_cfg["path"] = KAGGLE_DATA_ROOT
patched_data = Path("/kaggle/working/data_patched.yaml")
patched_data.write_text(yaml.safe_dump(data_cfg, sort_keys=False))

print("Hiperparámetros:", hyp)
print("data.yaml parcheado:", patched_data)

In [ ]:
# Entrenamiento. Usa la API de Ultralytics directamente; toda la
# justificación de hiperparámetros vive en el archivo YAML cargado.
from ultralytics import YOLO

model = YOLO(hyp["model"])
results = model.train(
    data=str(patched_data),
    name="exp_D_close_mosaic",
    project="/kaggle/working/runs",
    **{k: v for k, v in hyp.items() if k != 'model'},
)

# Los artefactos quedan en /kaggle/working/runs/<name>/. Descargar el
# best.pt al final del run y publicarlo como release del repo Git.

In [ ]:
# Listar artefactos producidos para descarga (no commitear los .pt).
from pathlib import Path
run_dir = Path("/kaggle/working/runs/exp_D_close_mosaic")
for p in sorted(run_dir.rglob('*')):
    if p.is_file():
        print(p)